In [1]:
import sqlite3
import sqlite_vec
import ollama
import pandas as pd
from transformers import AutoModel
import torch
from graphqlite import Graph

/Users/chielerli/Programming/Agent_Instructions/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
conn = sqlite3.connect('sqlite.db')
cursor = conn.cursor()
conn.enable_load_extension(True)
sqlite_vec.load(conn)
g = Graph("splite.db")


In [31]:
res = pd.read_sql_query("SELECT * FROM vector_fact_mem;", conn)
res

,id,node_embedding
0,1,"b'\x00\x00\xc0\xbd\x00\x00x<\x00\x00""<\x00\x00..."
1,2,b'\x00\x00\xd4\xbd\x00\x00z<\x00\x00\xc2<\x00\...
2,3,b'\x00\x00\xbd\xbd\x00\x00\x16\xbc\x00\x00\x92...
3,4,b'\x00\x00\x9f\xbb\x00\x00\xe5\xbc\x00\x00\x1d...
4,5,b'\x00\x00\x14\xbe\x00\x00\xa5<\x00\x00\t=\x00...
5,6,b'\x00\x00\xed\xbd\x00\x00W=\x00\x00p=\x00\x00...
6,7,b'\x00\x00o\xbd\x00\x00\x9b<\x00\x00\x90\xbd\x...
7,8,b'\x00\x00\xbe\xbd\x00\x00\xab\xbc\x00\x00J=\x...


In [9]:
res = conn.execute("""CREATE TABLE IF NOT EXISTS entities(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            entity TEXT NOT NULL,
            last_accessed TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
""")
conn.commit()
conn.close()

In [ ]:
# CREATE_ENT_TABLE_CMD="""
# CREATE TABLE IF NOT EXISTS entities(
#     id INTEGER PRIMARY KEY AUTOINCREMENT, --id matches to id in graphqlite
#     entity_id 
#     entity TEXT NOT NULL,
#     last_accessed TIMESTAMP DEFAULT CURRENT_TIMESTAMP
# );
# """

# CREATE_EPISODIC_LOG_CMD = """
# CREATE TABLE IF NOT EXISTS episodic_logs(
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
#     role TEXT NOT NULL, -- 'user', 'agent', or 'system_reflection'
#     content TEXT NOT NULL, -- The raw text/conversation turn
#     metadata TEXT -- JSON string for tags, session_ids, etc.
# )

# """

# CREATE_VECTOR_MEM_CMD="""
# CREATE VIRTUAL TABLE IF NOT EXISTS vector_mem using vec0(
#     id INTEGER PRIMARY KEY,
#     embeddings float[256]

# )



# CREATE_VECTOR_FACT_MEM_CMD = """
# CREATE VIRTUAL TABLE IF NOT EXISTS vector_fact_mem using vec0(
#     node_id INTEGER PRIMARY KEY,
#     node_embedding float[256]
# )
# """

In [7]:



model = AutoModel.from_pretrained(
    "jinaai/jina-embeddings-v5-text-nano",
    trust_remote_code=True,
    force_download=True,  # Forces a fresh copy of the code from HF Hub
    dtype=torch.bfloat16
)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device=device)

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 62953.91it/s]


In [ ]:
query_embedding = model.encode(texts=['Testing'], task = 'retrieval', prompt_name = "document")[0]
len(query_embedding)
query_embedding = query_embedding[:256].tolist()

In [47]:
res = pd.read_sql_query("SELECT * FROM vector_mem", conn)
res

,id,embeddings
0,1,b'\x00\x00\xd0\xbd\x00\x00\xd1\xbc\x00\x00O;\x...
1,2,b'\x00\x00\xd0\xbd\x00\x00\xd1\xbc\x00\x00O;\x...
2,3,b'\x00\x00\xef\xbd\x00\x00\x90\xbc\x00\x00P<\x...


In [ ]:
def store_episodic_log(usr_text:str, agent_respone:str):
    combined = f"User asked [f{usr_text}] agent respond [{agent_respone}]"
    


In [3]:
import json
import sqlite3
import uuid
from graphqlite import Graph
import hashlib
from ollama import Client
import pandas as pd
import spacy
import sqlite_vec
import torch
from transformers import AutoModel


class Memory_Manager:
    DB_URL = "/Users/chielerli/Programming/Agent_Instructions/agent_workspace/memory/sqlite.db"
    
    # 💡 Improved LLM prompt to heavily enforce JSON format compliance for a 1B model
    OLLAMA_SYS_PROMPT = """
    Extract computer science, tools, frameworks, and architecture terms from the User Text.
    Also extract relationships like: depends_on, uses, is_a, references, calls, part_of.
    
    Return the result ONLY as a raw valid JSON array of objects. Do not write markdown blocks or conversational text.
    Format example: [{"source": "python", "target": "ollama", "relationship": "uses"}]

    TEXT TO ANALYZE:
    """

    conn = sqlite3.connect(DB_URL)
    conn.enable_load_extension(True)
    cursor = None
    embedding_model = None
    nlp = None
    G = None
    client = None
    namespace = uuid.NAMESPACE_OID

    def __init__(self):
        sqlite_vec.load(self.conn)
        self.cursor = self.conn.cursor()
        
        model = AutoModel.from_pretrained(
            "jinaai/jina-embeddings-v5-text-nano",
            trust_remote_code=True,
            dtype=torch.bfloat16
        )
        device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
        self.embedding_model = model.to(device=device)
        self.nlp = spacy.load("en_core_web_sm")
        self.G = Graph(self.DB_URL)
        self.client = Client(host="127.0.0.1:11434")

    def check_entity_exists_table(self, entity):
        # Normalizing search lookup
        entity_clean = entity.casefold().strip()
        row = self.cursor.execute("SELECT entity_id FROM entities WHERE entity = ?", (entity_clean,)).fetchone()
        return row[0] if row else None
    
    def check_rel_exists_graph(self, source_id, target_id):
        return self.G.has_edge(source_id, target_id)

    def get_embeddings(self, input_str: str, prompt_name="document"):
        emb = self.embedding_model.encode(texts=[input_str], task='retrieval', prompt_name=prompt_name)[0]
        return emb[:256].tolist()

    def get_relationships(self, full_text: str):
        response = self.client.chat(
            model="llama3.2:1b",
            messages=[{"role": "user", "content": self.OLLAMA_SYS_PROMPT + full_text}],
            options={"temperature": 0.0} # Force determinism to keep JSON clean
        )
        try:
            # ✅ FIX: Properly parse JSON string back into a Python list
            clean_content = response["message"]["content"].strip()
            # Basic fallback if model outputs markdown code blocks
            if clean_content.startswith("```json"):
                clean_content = clean_content.split("```json")[1].split("```")[0].strip()
            elif clean_content.startswith("```"):
                clean_content = clean_content.split("```")[1].split("```")[0].strip()
            return json.loads(clean_content)
        except Exception as e:
            print(f"Failed to parse LLM JSON response: {e}")
            return []
    
    def get_node_id(self, source):
        df = self.search_related_facts(source, k=1)
        return df.iloc[0]['entity_id'] if not df.empty else None

    def insert_turn_log(self, role: str, content: str, metadata=None):
        raw_vec = self.get_embeddings(content)
        self.cursor.execute(
            "INSERT INTO episodic_logs (role, content, metadata) VALUES(?, ?, ?)",
            (role, content, metadata)
        )
        log_id = self.cursor.lastrowid
        vector = sqlite_vec.serialize_float32(raw_vec)
        self.cursor.execute(
            "INSERT INTO vector_mem (id, embeddings) VALUES(?, ?)",
            (log_id, vector)
        )
        self.commit_all()

    def insert_relationships_graph(self, relationships: list):
            for rel in relationships:
                source = rel['source'].casefold().strip()
                target = rel['target'].casefold().strip()
                relationship = rel['relationship'].casefold().strip()
                
                source_id = self.check_entity_exists_table(source)
                target_id = self.check_entity_exists_table(target)
                
                if source_id and target_id and self.G.has_edge(source_id, target_id):
                    continue
                
                if not source_id:
                    source_id = int(uuid.uuid5(self.namespace, source))
                    self.insert_entities_table([source], [source_id])
                    # CRITICAL: Commit immediately here so the table lock drops
                    self.conn.commit() 
                    self.G.upsert_node(source_id, {"name": source}, label="Entity")
                    
                if not target_id:
                    target_id = int(uuid.uuid5(self.namespace, target))
                    self.insert_entities_table([target], [target_id])
                    # CRITICAL: Commit immediately here so the table lock drops
                    self.conn.commit() 
                    self.G.upsert_node(target_id, {"name": target}, label="Entity")

                self.G.upsert_edge(source_id, target_id, rel_type=relationship)
                
            # Ensure final state is written
            self.commit_all()

    def insert_entities_table(self, entities: list, entity_ids: list, prompt_name="document"):
        for i in range(len(entities)):
            fact = entities[i].casefold().strip()
            entity_id = str(entity_ids[i])

            if not self.check_entity_exists_table(fact):
                # ✅ FIX: Keep cross-compatibility solid by binding directly with entity_id string
                self.cursor.execute("INSERT INTO entities (entity_id, entity) VALUES(?, ?)", (entity_id, fact))
                
                raw_vec = self.get_embeddings(fact, prompt_name=prompt_name)
                vector = sqlite_vec.serialize_float32(raw_vec)
                
                # ✅ FIX: Corrected column name from 'embeddings' to 'node_embedding' to match your schema
                self.cursor.execute(
                    "INSERT INTO vector_fact_mem (node_id, node_embedding) VALUES(?, ?)",
                    (entity_id, vector)
                )
        self.commit_all()

    def search_related_episodic_logs(self, input_str: str, k=3):
        raw_vec = self.get_embeddings(input_str, prompt_name="query")
        vector = sqlite_vec.serialize_float32(raw_vec)
        knn = """SELECT
                v.id,
                v.embeddings,
                e.content
                FROM vector_mem v
                JOIN episodic_logs e ON e.id = v.id
                WHERE embeddings MATCH (?)
                AND k = (?)"""
        return pd.read_sql_query(knn, self.conn, params=[vector, k])

    def search_related_facts(self, input_str: str, k=3):
        raw_vec = self.get_embeddings(input_str, prompt_name="query")
        vector = sqlite_vec.serialize_float32(raw_vec)
        
        # ✅ FIX: Matched correctly against 'node_embedding' and selected 'entity_id'
        knn = """SELECT
                e.entity_id,
                v.node_embedding,
                e.entity
                FROM vector_fact_mem v 
                JOIN entities e ON e.entity_id = v.node_id
                WHERE node_embedding MATCH (?)
                AND k = (?)"""
        return pd.read_sql_query(knn, self.conn, params=[vector, k])

    def search_related_rel_graph(self, source):
        source_id = self.get_node_id(source)
        if source_id:
            # Executing graph traversal using Graphqlite Cypher interface
            rows = self.G.connection.cypher(
                """
                MATCH (e: Entity {id: $source_id}) -[r]-(related: Entity) 
                WHERE related.id <> $source_id
                RETURN DISTINCT 
                    e.name AS source, 
                    TYPE(r) AS relationship, 
                    related.name AS target
                """, 
                parameters={"source_id": source_id}
            )
        else:
            rows = []
        return rows

    def normalize_graph_result(self, rel_rows):
        normalized_res = ""
        # ✅ FIX: Properly tracking loop indexes through typical sequence elements
        for i, row in enumerate(rel_rows):
            normalized_res += f"relationship {i}: {row['source']} {row['relationship']} {row['target']}\n"
        return normalized_res

    def commit_all(self):
        self.conn.commit()

In [ ]:
conn.close()

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 101067.57it/s]


Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 96698.65it/s]


In [22]:

import pandas as pd
df = pd.read_sql_query("SELECT name FROM sqlite_schema WHERE type='table' AND name NOT LIKE 'sqlite_%';", conn)
df

,name
0,episodic_logs
1,vector_mem
2,vector_mem_info
3,vector_mem_chunks
4,vector_mem_rowids
5,vector_mem_vector_chunks00
6,vector_fact_mem
7,vector_fact_mem_info
8,vector_fact_mem_chunks
9,vector_fact_mem_rowids


In [50]:
import json
from ollama import Client
client = Client(host="127.0.0.1:11434")
def get_entities(input:str):
        import ast
        import re
        ENTITY_SYS_MSG="""
                        GOAL:
                        Extract only strict computer science, programming languages, tools, frameworks, and technical software architecture terms alongside named entities from the User Text.
                        Return the result ONLY as a raw, valid array of lowercase strings. Do not write any introductory text, markdown formatting, or explanations. Only source, target, relationship.
                        Example: ["python", "ollama"]
                        USER TEXT:
                        """

        response = client.chat(model="llama3.2:1b",
            messages = [{"role":"user", "content": ENTITY_SYS_MSG+input}])
        if hasattr(response, 'message') and hasattr(response.message, 'content'):
            response_text = response.message.content
        elif isinstance(response, dict):
            response_text = response.get('message', {}).get('content', '')
        else:
            response_text = str(response)

        # 2. FIX: Since llama3.2:1b can output multiple lists or extra markdown chat,
        # use a regex to grab all valid [...] arrays safely instead of a blind literal_eval.
        try:
            # Find everything wrapped in square brackets
            found_arrays = re.findall(r'\[.*?\]', response_text, re.DOTALL)
            
            entities = []
            for array_str in found_arrays:
                try:
                    # json.loads is typically faster and safer for string arrays than ast
                    parsed_list = json.loads(array_str)
                    if isinstance(parsed_list, list):
                        entities.extend(parsed_list)
                except json.JSONDecodeError:
                    # Fallback to ast if the model used single quotes instead of double quotes
                    try:
                        parsed_list = ast.literal_eval(array_str)
                        if isinstance(parsed_list, list):
                            entities.extend(parsed_list)
                    except Exception:
                        continue

            # Clean, de-duplicate, and normalize everything to lowercase
            clean_entities = list(set([str(e).strip().casefold() for e in entities]))
            return clean_entities
        except Exception as e:
            print(f"[Entity Extraction Error] Failed to parse response text: {e}")
            return []
res = get_entities("python is cool, just like pyspark")
res


['pyspark', 'sql', 'csharp', 'java']

In [44]:
real_list

['python', 'pyspark', 'pyscript', 'ollama']